<a href="https://colab.research.google.com/github/dominiksakic/NETworkingMay/blob/main/25_nlp_pretrained_word_embeddings_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# NLP Sequence with pretrained word embeddings
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  12.3M      0  0:00:06  0:00:06 --:--:-- 15.0M


In [2]:
!rm -r aclImdb/train/unsup

In [3]:
import os, pathlib, shutil, random
from tensorflow import keras

# Extract data
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)

# Create sets
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [4]:
# Learning word embeddings
from tensorflow.keras import layers

max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length = max_length,
)

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [5]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2025-05-28 11:44:53--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-05-28 11:44:53--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-05-28 11:44:53--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [7]:
import numpy as np
path_to_glove_file = "glove.6B.100d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
  for line in f:
    word, coefs = line.split(maxsplit=1)
    coefs = np.fromstring(coefs, "f", sep=" ")
    embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


In [22]:
embedding_dim = 100
# 20.000 most common words from the training data
vocabulary = text_vectorization.get_vocabulary()
# mapping from words to their index
word_index = dict(zip(vocabulary, range(len(vocabulary))))

# Example
print(f'Index the word the: {word_index["the"]}')

Index the word the: 2


In [43]:
# Matrix for the GloVe Vector
embedding_matrix = np.zeros((max_tokens, embedding_dim))

for word, i in word_index.items():
  if i < max_tokens:
    embedding_vector = embeddings_index.get(word)
  if embedding_vector is not None:
    embedding_matrix[i] = embedding_vector

"""
a) Vocab -> truncated size of 600 AND a dictonairy of the 20.000 most common words.
0 = " ", 1 = the, etc.

b) GloVe -> Words associated with Vectors
the = [floats, floats, ... , floats]

c) a + b = embedding_matrix
We use the position and the word from the Vocab to retrieve the embeddings Vec.
1. 2, the (Vocab)
2. Glove(the) = [floats,... , floats]
3. matrix[2] = [floats,... , floats]

d) Use the weights as a Layer, and freeze it to not loose it!
"""

embedding_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False, # important to not loose the weights while training!
    mask_zero=True,)

In [39]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = embedding_layer(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
  loss="binary_crossentropy",
  metrics=["accuracy"])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │  2,000,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 64)        │     34,048 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,034,113 (7.76 MB)

 Trainable params: 34,113 (133.25 KB)

 Non-trainable params: 2,000,000 (7.63 MB)

In [41]:
callbacks = [
    keras.callbacks.ModelCheckpoint("glove_embeddings_sequence_model.keras",
                                    save_best_only=True)
]

model.fit(int_train_ds, validation_data=int_val_ds, epochs=10,
          callbacks=callbacks)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 463s 729ms/step - accuracy: 0.6289 - loss: 0.6297 - val_accuracy: 0.7946 - val_loss: 0.4624
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 452s 723ms/step - accuracy: 0.7842 - loss: 0.4695 - val_accuracy: 0.8276 - val_loss: 0.3973
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 473s 677ms/step - accuracy: 0.8161 - loss: 0.4083 - val_accuracy: 0.8400 - val_loss: 0.3698
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 418s 669ms/step - accuracy: 0.8390 - loss: 0.3746 - val_accuracy: 0.8548 - val_loss: 0.3435
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 439s 664ms/step - accuracy: 0.8563 - loss: 0.3443 - val_accuracy: 0.8326 - val_loss: 0.3628
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 462s 697ms/step - accuracy: 0.8663 - loss: 0.3209 - val_accuracy: 0.8594 - val_loss: 0.3228
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 437s 700ms/step - accuracy: 0.8726 - loss: 0.3039 - val_accuracy: 0.8724 - val_loss: 0.3092
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 415s 665ms/step - accuracy: 0.8829 -

In [42]:
model = keras.models.load_model("glove_embeddings_sequence_model.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 129s 162ms/step - accuracy: 0.8741 - loss: 0.2972
Test acc: 0.876
